# Tutorial about fluopy - io

Here we demonstrate input/output and serialization of datastructures in fluopy.

In [ ]:
import pickle
import tempfile
from pathlib import Path

import numpy as np

import fluopy
import fluopy.analysis as an
import fluopy.blinking as bl
import fluopy.emissions as em
import fluopy.fcs as fcs_p
import fluopy.fluorophores as fl
import fluopy.miscellaneous as mi
import fluopy.prediction as pr
import fluopy.simulation as si
import fluopy.transitions as tr

In [ ]:
# ruff: noqa: S101, S301, S403

In [ ]:
fluopy.__version__

For random number generation a Generator is initialized.

In [ ]:
rng = np.random.default_rng(seed=1)

## Single dye simulation setup

For further demonstration we set up a simple fluorophore system that contains a single Cy5 fluorophore.

In [ ]:
%%time
fluorophore = fl.Fluorophore(name="cy5_dna", position=[0, 0])
fluorophore_system = fl.FluorophoreSystem(fluorophores=[fluorophore])

transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2,
    wavelength=640,
    bleaching=False,
    energy_transfer=False,
    dstorm=False,
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.remove_energy_transfers()
transition_set.finalize()

prediction = pr.Prediction(transition_set)
simulation = si.Simulation(transition_set)
simulation.run(start_at=None, size=1e6, end_time=None, seed=rng, use_memmap=None)

analysis = an.Analysis(simulation=simulation)
emissions = em.Emissions(frame_time="5ms", seed=rng, bandpass=(600, 800))
emissions.extract(simulation=simulation)
fcs = fcs_p.FCS(emissions)
blinks = bl.Blinking(emissions, threshold=5)

## Pickle

To save data temporarily in a python environment, the data classes can be serialized (pickled), saved and reloaded.

### Pickle TransitionSet

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "transition_set.pickle"

    with open(file_path, mode="wb") as file:
        pickle.dump(transition_set, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        transition_set_new = pickle.load(file)

assert transition_set.transition_df.equals(transition_set_new.transition_df)
transition_set_new.transition_df

### Pickle Prediction

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "prediction.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(prediction, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        prediction_new = pickle.load(file)

assert np.array_equal(
    prediction.mean_transition_times, prediction_new.mean_transition_times
)
mi.print_class(prediction_new)

### Pickle Simulation

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "simulation.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(simulation, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        simulation_new = pickle.load(file)

assert np.array_equal(simulation.transition_series, simulation_new.transition_series)
mi.print_class(simulation_new)

### Pickle Emissions

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "emissions.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(emissions, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        emissions_new = pickle.load(file)


assert emissions.event_time_series.equals(emissions_new.event_time_series)
mi.print_class(emissions_new)

### Pickle Analysis

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "analysis.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(analysis, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        analysis_new = pickle.load(file)

assert np.array_equal(
    analysis.mean_transition_times, analysis_new.mean_transition_times
)
mi.print_class(analysis_new)

### Pickle FCS

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "fcs.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(fcs, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        fcs_new = pickle.load(file)

assert np.array_equal(fcs.autocorrelation, fcs_new.autocorrelation)
mi.print_class(fcs_new)

### Pickle Blinks

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = Path(tmpdirname) / "blinks.pickle"
    print("created temporary", file_path)

    with open(file_path, mode="wb") as file:
        pickle.dump(blinks, file=file, protocol=pickle.HIGHEST_PROTOCOL)

    with open(file_path, mode="rb") as file:
        blinks_new = pickle.load(file)

assert np.array_equal(blinks.on_periods, blinks_new.on_periods)
mi.print_class(blinks_new)